## Speed Up Your Python Program With Concurrency

Concurrency refers to the ability of a program to manage multiple tasks at once, improving performance and responsiveness. It encompasses different models: 
- threading (Thread)
- asynchronous tasks (Task)
- multiprocessing (Process)

Each offering unique benefits and trade-offs. In Python, threads and asynchronous tasks facilitate concurrency on a single processor, while ***multiprocessing allows for true parallelism*** by utilizing multiple CPU cores.

See:
- https://realpython.com/python-concurrency/
- https://realpython.com/intro-to-python-threading/

## Speeding Up an I/O-Bound Program

An I/O-bound problem spends most of its time waiting for external operations to complete, such as network calls.


### Synchronous Version
This version of your program doesn’t use concurrency at all.

In [4]:
import time

import requests

def main():
    sites = [
        "https://www.jython.org",
        "http://olympus.realpython.org/dice",
    ] * 10
    start_time = time.perf_counter()
    download_all_sites(sites)
    duration = time.perf_counter() - start_time
    print(f"Downloaded {len(sites)} sites in {duration} seconds")

def download_all_sites(sites):
    '''
     It’s possible to call requests.get() directly, but creating a Session object allows 
     the library to retain state across requests and reuse the connection to speed things up.
    '''
    with requests.Session() as session:
        for url in sites:
            download_site(url, session)

def download_site(url, session):
    with session.get(url) as response:
        print(f"Read {len(response.content)} bytes from {url}")

if __name__ == "__main__":
    main()

Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice
Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice
Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice
Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice
Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice
Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice
Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice
Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice
Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice
Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice


### Multi-Threaded Version
Using concurrent.futures and threading modules. A Thread-Pool-Executor, you’ll end up with these three components:
- Thread
- Pool
- Executor

In [4]:
import threading
import time
from concurrent.futures import ThreadPoolExecutor

import requests

# The threading.local() function creates a thread-local storage (TLS) object, which allows you to 
# store data that is isolated and unique to each individual thread. Even if multiple threads share 
# the same threading.local instance, the values they assign to it are not visible to one another.
thread_local = threading.local()

def main():
    sites = [
        "https://www.jython.org",
        "http://olympus.realpython.org/dice",
    ] * 10
    start_time = time.perf_counter()
    download_all_sites(sites)
    duration = time.perf_counter() - start_time
    print(f"Downloaded {len(sites)} sites in {duration} seconds")

def download_all_sites(sites):
    # Create a pool of 5 threads for executing a function
    with ThreadPoolExecutor(max_workers=5) as executor:
        executor.map(download_site, sites)

def download_site(url):
    session = get_session_for_thread()
    with session.get(url) as response:
        print(f"Read {len(response.content)} bytes from {url}")

def get_session_for_thread():
    if not hasattr(thread_local, "session"):
        thread_local.session = requests.Session()
    return thread_local.session

if __name__ == "__main__":
    main()

Read 10966 bytes from https://www.jython.org
Read 10966 bytes from https://www.jython.org
Read 10966 bytes from https://www.jython.org
Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice
Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice
Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice
Read 272 bytes from http://olympus.realpython.org/dice
Read 272 bytes from http://olympus.realpython.org/dice
Read 272 bytes from http://olympus.realpython.org/dice
Read 10966 bytes from https://www.jython.org
Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice
Read 272 bytes from http://olympus.realpython.org/dice
Read 10966 bytes from https://www.jython.org
Read 10966 bytes from https://www.jython.org
Read 272 bytes from http://olympus.realpython.org/dice
Read 272 bytes from http://olympus.realpython.org/dice


### Asynchronous Version

Using asyncio (Async IO) and aiohttp (Async IO HTTP) library.

See https://realpython.com/python-concurrency/#asynchronous-version

### Process-Based Version

Using ProcessPoolExecutor

See https://realpython.com/python-concurrency/#process-based-version

The multiprocessing module, along with the corresponding wrappers in concurrent.futures, was designed to break down single CPU barrier and run your code across multiple CPUs. At a high level, it does this by creating a new instance of the Python interpreter to run on each CPU and then farming out part of your program to run on it.


In [3]:
import atexit
import multiprocessing
import time
from concurrent.futures import ProcessPoolExecutor

import requests

session: requests.Session

def main():
    sites = [
        "https://www.jython.org",
        "http://olympus.realpython.org/dice",
    ] * 10
    start_time = time.perf_counter()
    download_all_sites(sites)
    duration = time.perf_counter() - start_time
    print(f"Downloaded {len(sites)} sites in {duration} seconds")

def download_all_sites(sites):
    with ProcessPoolExecutor(initializer=init_process) as executor:
        executor.map(download_site, sites)

def download_site(url):
    with session.get(url) as response:
        name = multiprocessing.current_process().name
        print(f"{name}:Read {len(response.content)} bytes from {url}")

def init_process():
    global session
    session = requests.Session()
    atexit.register(session.close)

if __name__ == "__main__":
    main()

ForkProcess-33:Read 10966 bytes from https://www.jython.org
ForkProcess-45:Read 10966 bytes from https://www.jython.orgForkProcess-42:Read 272 bytes from http://olympus.realpython.org/diceForkProcess-35:Read 272 bytes from http://olympus.realpython.org/diceForkProcess-48:Read 272 bytes from http://olympus.realpython.org/diceForkProcess-44:Read 272 bytes from http://olympus.realpython.org/dice




ForkProcess-38:Read 272 bytes from http://olympus.realpython.org/diceForkProcess-37:Read 10966 bytes from https://www.jython.orgForkProcess-46:Read 272 bytes from http://olympus.realpython.org/diceForkProcess-40:Read 272 bytes from http://olympus.realpython.org/dice



ForkProcess-36:Read 272 bytes from http://olympus.realpython.org/dice
ForkProcess-33:Read 10966 bytes from https://www.jython.org
ForkProcess-39:Read 10966 bytes from https://www.jython.orgForkProcess-43:Read 10966 bytes from https://www.jython.org

ForkProcess-47:Read 10966 bytes from https://www.jython.orgForkProcess-41:Read 1